# WHL Season Simulation Notebook

This notebook guides you through the WHL simulation pipeline:
1. Loading raw shift data.
2. Building matchup features.
3. Computing ELO ratings.
4. Training goal prediction models.
5. Simulating seasons.
6. Analyzing final standings.

In [ ]:
# Basic setup
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src/ to path if needed
BASE_DIR = Path('../')
sys.path.insert(0, str(BASE_DIR / 'src'))

from database import create_tables, load_csv_to_db
from features import build_line_matchup_features
from elo import EloSystem
from lgbm_model import LightGBMGoalModel
from simulator import SeasonSimulator
from ranking import compute_standings

DATA_DIR = BASE_DIR / 'data'
RAW_FILE = DATA_DIR / 'raw' / 'whl25.csv'
DB_FILE = DATA_DIR / 'whl.db'

## 1. Load Raw Data into Database

In [ ]:
# Create DB and load CSV
if DB_FILE.exists():
    print(f"[INFO] Database exists, removing {DB_FILE}")
    DB_FILE.unlink()

create_tables()
load_csv_to_db(RAW_FILE)
print("[INFO] Data loaded into SQLite database")

## 2. Build Features and Filter Data

In [ ]:
df = build_line_matchup_features()
df = df[df['toi'] > 0]  # remove zero time-on-ice rows

print(f"[INFO] Feature DataFrame shape: {df.shape}")
df.head()

## 3. Compute ELO Ratings

In [ ]:
elo = EloSystem()
games = df.groupby(['home_team','away_team']).agg({'home_xg':'sum','away_xg':'sum'}).reset_index()

for _, row in games.iterrows():
    elo.update(row['home_team'], row['away_team'], row['home_xg'], row['away_xg'])

elo_df = pd.DataFrame.from_dict(elo.ratings, orient='index', columns=['ELO']).sort_values('ELO', ascending=False)
elo_df.head()

## 4. Prepare Training Data for LightGBM

In [ ]:
df['elo_home'] = df['home_team'].map(elo.ratings)
df['elo_away'] = df['away_team'].map(elo.ratings)
df['elo_diff'] = df['elo_home'] - df['elo_away']

features = ['home_xg', 'away_xg', 'toi', 'xg_per_60', 'goals_per_60', 'elo_home', 'elo_away', 'elo_diff']
X = df[features]
y_home = df['home_goals']
y_away = df['away_goals']

print(f"[INFO] Training features shape: {X.shape}")

## 5. Train LightGBM Goal Models

In [ ]:
model_home = LightGBMGoalModel()
model_home.fit(X, y_home)

model_away = LightGBMGoalModel()
model_away.fit(X, y_away)

print("[INFO] Goal models trained")

## 6. Simulate a Single Season

In [ ]:
simulator = SeasonSimulator(model_home, model_away, elo)
season_results = []

for _, row in games.iterrows():
    features_home = pd.DataFrame([{ 
        'home_xg': row['home_xg'], 'away_xg': row['away_xg'], 'toi': row['home_xg']+row['away_xg'],
        'xg_per_60': row['home_xg']/max(1,row['home_xg']+row['away_xg'])*3600,
        'goals_per_60': 0,
        'elo_home': elo.ratings[row['home_team']],
        'elo_away': elo.ratings[row['away_team']],
        'elo_diff': elo.ratings[row['home_team']] - elo.ratings[row['away_team']]
    }])
    features_away = pd.DataFrame([{ 
        'home_xg': row['away_xg'], 'away_xg': row['home_xg'], 'toi': row['home_xg']+row['away_xg'],
        'xg_per_60': row['away_xg']/max(1,row['home_xg']+row['away_xg'])*3600,
        'goals_per_60': 0,
        'elo_home': elo.ratings[row['away_team']],
        'elo_away': elo.ratings[row['home_team']],
        'elo_diff': elo.ratings[row['away_team']] - elo.ratings[row['home_team']]
    }])
    
    g_home, g_away, went_ot = simulator.simulate_game(row['home_team'], row['away_team'], features_home, features_away)
    season_results.append((row['home_team'], row['away_team'], g_home, g_away, went_ot))

standings = compute_standings(season_results)
pd.DataFrame([{'Team': t, 'Points': s['points'], 'GF': s['gf'], 'GA': s['ga']} for t,s in standings])

## 7. Visualize ELO Ratings and Standings

In [ ]:
# Plot ELO
plt.figure(figsize=(12,6))
elo_df['ELO'].sort_values().plot(kind='barh', color='steelblue')
plt.title('Team ELO Ratings')
plt.xlabel('ELO')
plt.show()

In [ ]:
# Plot simulated standings
teams, points = zip(*[(team, stats['points']) for team, stats in standings])
plt.figure(figsize=(12,6))
plt.barh(teams, points, color='orange')
plt.xlabel('Points')
plt.title('Simulated Season Standings')
plt.show()